# Data preprocessing
This notebook takes care of preprocessing the data, including:
- session construction for the 1k dataset
- filtering and vocabulary
- splitting both 1k and 360k datasets into train/test/val
- negative sampling
- exporting processed files

### Setups and imports

In [29]:
import pandas as pd
import numpy as np
import pickle
import os
import torch
from sklearn.model_selection import train_test_split
import random

In [30]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [31]:
DATA_DIR = "datasets/"
GAP_MIN = 30 # gap between tracks to start a session (in minutes)
MIN_SEQ_LENGTH = 5 # keep only tracks that appear at least MIN_ITEM_COUNT times
MAX_SEQ_LENGTH = 50 # maximum number of tracks per session
NUM_NEGATIVES = 100 # number of negative samples per positive interaction

### Loading raw data

In [32]:
# 1K dataset
PATH_1K = DATA_DIR + "lastfm-dataset-1K/userid-timestamp-artid-artname-traid-traname.tsv"
COLS_1K = ["user_id", "timestamp", "artist_id", "artist_name", "track_id", "track_name"]
df_1k = pd.read_csv(
    PATH_1K,
    sep='\t',
    names=COLS_1K,
    on_bad_lines='skip',
    quoting=3,
)
df_1k.dropna(inplace=True)
df_1k.drop_duplicates(inplace=True)

print("1K dataset shape:", df_1k.shape)
df_1k["timestamp"] = pd.to_datetime(df_1k["timestamp"])
df_1k = df_1k.sort_values(by=["user_id", "timestamp"])
df_1k.head()

1K dataset shape: (16982097, 6)


,user_id,timestamp,artist_id,artist_name,track_id,track_name
16684,user_000001,2006-08-13 13:59:20+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,c4633ab1-e715-477f-8685-afa5f2058e42,The Launching Of Big Face
16683,user_000001,2006-08-13 14:03:29+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,bc2765af-208c-44c5-b3b0-cf597a646660,Zn Zero
16682,user_000001,2006-08-13 14:10:43+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,aa9c5a80-5cbe-42aa-a966-eb3cfa37d832,The Return Of Super Barrio - End Credits
16681,user_000001,2006-08-13 14:17:40+00:00,67fb65b5-6589-47f0-9371-8a40eb268dfb,Tommy Guerrero,d9b1c1da-7e47-4f97-a135-77260f2f559d,Mission Flats
16680,user_000001,2006-08-13 14:19:06+00:00,1cfbc7d1-299c-46e6-ba4c-1facb84ba435,Artful Dodger,120bb01c-03e4-465f-94a0-dce5e9fac711,What You Gonna Do?


In [33]:
# 350K dataset
PATH_360K = DATA_DIR + "lastfm-dataset-360K/usersha1-artmbid-artname-plays.tsv"
COLS_360K = ["user_id", "artist_mbid", "artist_name", "plays"]

df_360k = pd.read_csv(
    PATH_360K,
    sep='\t',
    names=COLS_360K,
    on_bad_lines='skip',
    quoting=3,
)
df_360k.dropna(inplace=True)
df_360k.drop_duplicates(inplace=True)

print("360K dataset shape:", df_360k.shape)
df_360k.head()

360K dataset shape: (17332876, 4)


,user_id,artist_mbid,artist_name,plays
0,00000c289a1829a808ac09c00daf10bc3c4e223b,3bd73256-3905-4f3a-97e2-8b341527f805,betty blowtorch,2137
1,00000c289a1829a808ac09c00daf10bc3c4e223b,f2fb0ff0-5679-42ec-a55c-15109ce6e320,die Ärzte,1099
2,00000c289a1829a808ac09c00daf10bc3c4e223b,b3ae82c2-e60b-4551-a76d-6620f1b456aa,melissa etheridge,897
3,00000c289a1829a808ac09c00daf10bc3c4e223b,3d6bbeb7-f90e-4d10-b440-e153c0d10b53,elvenking,717
4,00000c289a1829a808ac09c00daf10bc3c4e223b,bbd2ffd7-17f4-4506-8572-c1ea58c3f9a8,juliette & the licks,706


remove users with very little interactions in the 360k dataset:

In [34]:
user_counts = df_360k['user_id'].value_counts()

threshold = 20
keep_users = user_counts[user_counts >= threshold].index
df_360k=df_360k[df_360k['user_id'].isin(keep_users)]

In [35]:
df_360k[df_360k['user_id'].isin(keep_users)]

,user_id,artist_mbid,artist_name,plays
0,00000c289a1829a808ac09c00daf10bc3c4e223b,3bd73256-3905-4f3a-97e2-8b341527f805,betty blowtorch,2137
1,00000c289a1829a808ac09c00daf10bc3c4e223b,f2fb0ff0-5679-42ec-a55c-15109ce6e320,die Ärzte,1099
2,00000c289a1829a808ac09c00daf10bc3c4e223b,b3ae82c2-e60b-4551-a76d-6620f1b456aa,melissa etheridge,897
3,00000c289a1829a808ac09c00daf10bc3c4e223b,3d6bbeb7-f90e-4d10-b440-e153c0d10b53,elvenking,717
4,00000c289a1829a808ac09c00daf10bc3c4e223b,bbd2ffd7-17f4-4506-8572-c1ea58c3f9a8,juliette & the licks,706
...,...,...,...,...
17559525,"sep 20, 2008",7ffd711a-b34d-4739-8aab-25e045c246da,turbostaat,12
17559526,"sep 20, 2008",9201190d-409f-426b-9339-9bd7492443e2,cuba missouri,11
17559527,"sep 20, 2008",e7cf7ff9-ed2f-4315-aca8-bcbd3b2bfa71,little man tate,11
17559528,"sep 20, 2008",f6f2326f-6b25-4170-b89d-e235b25508e8,sigur rós,10


In [36]:
df_360k

,user_id,artist_mbid,artist_name,plays
0,00000c289a1829a808ac09c00daf10bc3c4e223b,3bd73256-3905-4f3a-97e2-8b341527f805,betty blowtorch,2137
1,00000c289a1829a808ac09c00daf10bc3c4e223b,f2fb0ff0-5679-42ec-a55c-15109ce6e320,die Ärzte,1099
2,00000c289a1829a808ac09c00daf10bc3c4e223b,b3ae82c2-e60b-4551-a76d-6620f1b456aa,melissa etheridge,897
3,00000c289a1829a808ac09c00daf10bc3c4e223b,3d6bbeb7-f90e-4d10-b440-e153c0d10b53,elvenking,717
4,00000c289a1829a808ac09c00daf10bc3c4e223b,bbd2ffd7-17f4-4506-8572-c1ea58c3f9a8,juliette & the licks,706
...,...,...,...,...
17559525,"sep 20, 2008",7ffd711a-b34d-4739-8aab-25e045c246da,turbostaat,12
17559526,"sep 20, 2008",9201190d-409f-426b-9339-9bd7492443e2,cuba missouri,11
17559527,"sep 20, 2008",e7cf7ff9-ed2f-4315-aca8-bcbd3b2bfa71,little man tate,11
17559528,"sep 20, 2008",f6f2326f-6b25-4170-b89d-e235b25508e8,sigur rós,10


### Vocabulary
The reason for the vocabulary is that all the IDs are either alphanumerical or discontinuous, not suitable to be fed into a neural network.
Two vocabularies are created:
- track2idx, idx2track: Built only on the **1K dataset**, the 360K does not have track specific info
- artist2idx, idx2artist: Build on both 1K and 360K datasets
- user2idx, idx2user: Build only on the **360K dataset**, because the datasets use different user IDs, but the 1K is used for sequential and transformer models, which do not need the user ID.

Since many tracks and artists do not have an ID, it was necessary to create a custom ID as vocabulary key, by merging track ID with track name, or artist ID with artist name.

In [ ]:
# 1. User keys
df_360k['user_key'] = df_360k['user_id'].astype(str)
df_1k['user_key'] = df_1k['user_id'].astype(str)

# 2. Artist Keys (Artist ID + Artist Name)

df_1k['artist_key'] = df_1k['artist_id'].fillna(df_1k['artist_name']) 
df_1k['artist_key'] = (
    df_1k['artist_key']
    .fillna("unknown")
    .str.lower()
    .str.strip()
    .str.replace(r"[^a-z0-9]+", "", regex=True)
)
df_360k['artist_key'] = df_360k['artist_id'].fillna(df_360k['artist_name']) 
df_360k['artist_key'] = (
    df_1k['artist_key']
    .fillna("unknown")
    .str.lower()
    .str.strip()
    .str.replace(r"[^a-z0-9]+", "", regex=True)
)
# 3. Track Keys (Track ID + Track Name)
df_1k['track_key'] = df_1k['track_id'].fillna('no_id') + ":::" + df_1k['track_name'].fillna('unknown')

In [65]:
def build_vocab(keys):
    sorted_keys = sorted(set(keys)) # for reproducibility

    item2idx = {item: i + 1 for i, item in enumerate(sorted_keys)}
    item2idx['<PAD>'] = 0

    idx2item = {i + 1: item for i, item in enumerate(sorted_keys)}
    idx2item['<PAD>'] = 0

    return item2idx, idx2item

user2idx, idx2user = build_vocab(df_360k["user_key"])
# merging both dataset artist keys to create a shared dict
artist2idx, idx2artist = build_vocab(list(set(df_1k["artist_key"]) | set(df_360k["artist_key"])))
track2idx, idx2track = build_vocab(df_1k["track_key"])

In [66]:
len(track2idx)

961604

### Substituting the default IDs with my vocab IDs

In [67]:
df_1k_map = df_1k.copy()
df_1k_map["artist_key"] = df_1k_map["artist_key"].map(artist2idx)
df_1k_map["track_key"] = df_1k_map["track_key"].map(track2idx)
df_1k_map.head()

KeyboardInterrupt: 

In [ ]:
df_360k_map = df_360k.copy()
df_360k_map["user_key"] = df_360k_map["user_key"].map(user2idx)
df_360k_map["artist_key"] = df_360k_map["artist_key"].map(artist2idx)

df_360k_map = df_360k_map.drop(columns=["user_id", "artist_mbid"])
df_360k_map = df_360k_map[["user_key", "artist_key", "artist_name", "plays"]]

df_360k_map.head()

,user_key,artist_key,artist_name,plays
0,49,41594,betty blowtorch,2137
1,49,168751,die Ärzte,1099
2,49,124750,melissa etheridge,897
3,49,42727,elvenking,717
4,49,130359,juliette & the licks,706


### Session construction (1K only)

In [ ]:
# creating sessions for 1K dataset, gap between tracks is GAP_MIN minutes
df_1k_map["time_diff"] = df_1k_map.groupby("user_id")["timestamp"].diff()

gap_threshold = pd.Timedelta(minutes=GAP_MIN)
df_1k_map["new_session"] = (df_1k_map["time_diff"] > gap_threshold) | (df_1k_map["time_diff"].isna())

df_1k_map["session_id"] = df_1k_map.groupby("user_id")["new_session"].cumsum()

df_1k_map['global_session_id'] = df_1k_map['user_id'].astype(str) + "_" + df_1k_map['session_id'].astype(str)

# filter short and longs sessions
session_lengths = df_1k_map.groupby('global_session_id').size()

valid_sessions = session_lengths[(session_lengths >= MIN_SEQ_LENGTH) & (session_lengths <= MAX_SEQ_LENGTH)].index
df_1k_clean = df_1k_map[df_1k_map['global_session_id'].isin(valid_sessions)]

df_1k_clean.head()

,user_id,timestamp,artist_id,artist_name,track_id,track_name,user_key,artist_key,track_key,time_diff,new_session,session_id,global_session_id
16684,user_000001,2006-08-13 13:59:20+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,c4633ab1-e715-477f-8685-afa5f2058e42,The Launching Of Big Face,user_000001,6686,737481,NaT,True,1,user_000001_1
16683,user_000001,2006-08-13 14:03:29+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,bc2765af-208c-44c5-b3b0-cf597a646660,Zn Zero,user_000001,6686,706424,0 days 00:04:09,False,1,user_000001_1
16682,user_000001,2006-08-13 14:10:43+00:00,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,aa9c5a80-5cbe-42aa-a966-eb3cfa37d832,The Return Of Super Barrio - End Credits,user_000001,6686,640305,0 days 00:07:14,False,1,user_000001_1
16681,user_000001,2006-08-13 14:17:40+00:00,67fb65b5-6589-47f0-9371-8a40eb268dfb,Tommy Guerrero,d9b1c1da-7e47-4f97-a135-77260f2f559d,Mission Flats,user_000001,72330,817359,0 days 00:06:57,False,1,user_000001_1
16680,user_000001,2006-08-13 14:19:06+00:00,1cfbc7d1-299c-46e6-ba4c-1facb84ba435,Artful Dodger,120bb01c-03e4-465f-94a0-dce5e9fac711,What You Gonna Do?,user_000001,20157,67722,0 days 00:01:26,False,1,user_000001_1


In [ ]:
sessions = df_1k_clean.groupby("global_session_id").agg({
    "user_key": "first",
    "artist_key": list,
    "track_key": list,
    "timestamp": "min"
})
sessions = sessions.rename(columns={'timestamp': 'session_start'})
sessions.head()

,user_key,artist_key,track_key,session_start
global_session_id,,,,
user_000001_1,user_000001,"[6686, 6686, 6686, 72330, 20157, 74821, 74821,...","[737481, 706424, 640305, 817359, 67722, 448287...",2006-08-13 13:59:20+00:00
user_000001_10,user_000001,"[137514, 77384, 134414, 170784, 151221]","[441013, 460977, 198341, 434974, 240381]",2006-08-21 17:55:52+00:00
user_000001_100,user_000001,"[168181, 168181, 19745, 19745, 19745, 19745, 1...","[155170, 744218, 226370, 527575, 536705, 28745...",2006-11-16 01:49:47+00:00
user_000001_1000,user_000001,"[16886, 16886, 16886, 16886, 16886, 16886, 168...","[90123, 215003, 168408, 283808, 841571, 90123,...",2009-04-05 14:52:39+00:00
user_000001_1001,user_000001,"[16886, 16886, 16886, 16886, 16886, 16886, 168...","[652847, 840262, 107907, 90123, 215003, 840262...",2009-04-06 17:38:45+00:00


### Train/Val/Test split
Now the dataset get split into training, validation and test set.
- The 360K dataset is divided randomly by user, following a 80/10/10 rule
- The 1K dataset instead, requires more carefulness. The last session of each user is used as test set, the second to last as validation set, the rest is the training set. This way, the dataset is split chronologically.

In [ ]:
sessions['rank'] = sessions.groupby('user_key')['session_start'].rank(
    method='first', 
    ascending=False
).astype(int)

test_1k_set = sessions[sessions['rank'] == 1].drop(columns=['rank'])

val_1k_set = sessions[sessions['rank'] == 2].drop(columns=['rank'])

train_1k_set = sessions[sessions['rank'] > 2].drop(columns=['rank'])

print(f"Split complete:\nTrain: {len(train_1k_set)}\nVal: {len(val_1k_set)}\nTest: {len(test_1k_set)}")

Split complete:
Train: 547940
Val: 980
Test: 987


In [ ]:
train_users = set(train_1k_set["user_key"])
val_users = set(val_1k_set["user_key"])

missing_users = val_users - train_users
print(f"Users in val but not in train: {len(missing_users)}")

Users in val but not in train: 5


In [ ]:
train_360k_set, temp = train_test_split(
    df_360k_map,
    test_size=0.2,
    random_state=42,
    stratify=None
)

val_360k_set, test_360k_set = train_test_split(
    temp,
    test_size=0.5,
    random_state=42
)

print("Split complete")
print(f"Train: {len(train_360k_set)} rows")
print(f"Val:   {len(val_360k_set)} rows")
print(f"Test:  {len(test_360k_set)} rows")

Split complete
Train: 13835268 rows
Val:   1729408 rows
Test:  1729409 rows


for validating and testing the matrix factorization, I need known users:

In [63]:
def user_stratified_split(df, test_size=0.2):
    train_list = []
    val_list=[]
    test_list = []
    
    for user, group in df.groupby('user_key'):
        if len(group) < 3:
            train_list.append(group)
            continue
            
        u_train, u_temp = train_test_split(
            group, 
            test_size=test_size, 
            random_state=42
        )
        if len(u_temp) < 2:
            u_val = u_temp
            u_test = pd.DataFrame(columns=df.columns)
        else:
            u_val, u_test = train_test_split(
                u_temp, 
                test_size=0.5, 
                random_state=42
            )

        train_list.append(u_train)
        val_list.append(u_val)
        test_list.append(u_test)
        
    return pd.concat(train_list),pd.concat(val_list), pd.concat(test_list)

train_360k_set,val_360k_set, test_360k_set = user_stratified_split(df_360k_map)

KeyboardInterrupt: 

### Saving datasets and vocabs

In [ ]:
os.makedirs("datasets/processed", exist_ok=True)

os.makedirs("datasets/processed/test", exist_ok=True)
os.makedirs("datasets/processed/train", exist_ok=True)
os.makedirs("datasets/processed/val", exist_ok=True)


TRAIN_PATH = "datasets/processed/train"
VAL_PATH = "datasets/processed/val"
TEST_PATH = "datasets/processed/test"

# Train
train_1k_set.to_pickle(TRAIN_PATH + "/session_train.pkl")
train_360k_set.to_pickle(TRAIN_PATH + "/train_360K.pkl")

# Val
val_1k_set.to_pickle(VAL_PATH + "/session_val.pkl")
val_360k_set.to_pickle(VAL_PATH + "/val_360K.pkl")

# Test
test_1k_set.to_pickle(TEST_PATH + "/session_test.pkl")
test_360k_set.to_pickle(TEST_PATH + "/test_360K.pkl")


In [ ]:
with open("datasets/processed/user_vocab.pkl", "wb") as f:
    pickle.dump({"user2idx": user2idx, "idx2user": idx2user}, f)

with open("datasets/processed/track_vocab.pkl", "wb") as f:
    pickle.dump({"track2idx": track2idx, "idx2track": idx2track}, f)

with open("datasets/processed/artist_vocab.pkl", "wb") as f:
    pickle.dump({"artist2idx": artist2idx, "idx2artist": idx2artist}, f)